# 🚦 Smart-Traffic-MARL — Colab Training

Train GAT-MARL / IDQN trên Google Colab với SUMO simulator.

**Workflow:**
1. Cài SUMO + dependencies
2. Clone repo từ GitHub
3. Mount Google Drive để lưu checkpoint + log
4. Train (hỗ trợ resume nếu bị ngắt)
5. Download kết quả

> ⚠️ **Colab free tier**: max ~12h/session, 2 CPU cores → dùng 1 worker.
> **Colab Pro**: max ~24h, 4 cores → dùng 2 workers.

---
## 1. Cài đặt SUMO Simulator

In [ ]:
%%bash
# Cài SUMO từ Ubuntu repo — headless mode (không cần GUI)
apt-get update -qq
apt-get install -y -qq sumo sumo-tools > /dev/null 2>&1

# Verify
sumo --version | head -1
echo "✅ SUMO installed successfully"

In [ ]:
import os
os.environ["SUMO_HOME"] = "/usr/share/sumo"
print(f"SUMO_HOME = {os.environ['SUMO_HOME']}")

---
## 2. Clone repo + cài Python dependencies

In [ ]:
import os

REPO_URL = "https://github.com/MinhCYB/Traffic-MARL.git"
REPO_DIR = "/content/Traffic-MARL"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    print(f"✅ Cloned → {REPO_DIR}")
else:
    !cd {REPO_DIR} && git pull
    print(f"✅ Pulled latest → {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"📂 Working directory: {os.getcwd()}")

In [ ]:
# Cài Python dependencies
# torch thường đã có sẵn trên Colab, chỉ cần torch-geometric + traci
!pip install -q torch-geometric traci sumolib

# Verify imports
import torch
import traci
print(f"✅ PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
print(f"✅ TraCI ready")

---
## 3. Mount Google Drive (lưu checkpoint + log)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Tạo thư mục lưu trữ trên Drive
DRIVE_DIR = "/content/drive/MyDrive/Traffic-MARL-Training"
os.makedirs(f"{DRIVE_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/logs", exist_ok=True)
print(f"✅ Drive output dir: {DRIVE_DIR}")

---
## 4. Cấu hình Training

Chỉnh các tham số bên dưới trước khi chạy.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║                    TRAINING CONFIG                          ║
# ╚══════════════════════════════════════════════════════════════╝

MODEL       = "gat_marl"   # "gat_marl" hoặc "idqn"
NUM_WORKERS = 1            # Colab free: 1, Pro: 2
EPISODES    = 500          # Số episodes mỗi worker
DELTA_TIME  = 5            # Giây giữa mỗi decision step

# Obstacle (vật cản)
OBSTACLE_PROB     = 0.4    # Xác suất có vật cản mỗi episode
OBSTACLE_MAX      = 3      # Tối đa vật cản đồng thời
OBSTACLE_DUR_MIN  = 300    # Thời gian tối thiểu (giây)
OBSTACLE_DUR_MAX  = 600    # Thời gian tối đa (giây), None = xuyên suốt

# Resume từ checkpoint trước (để trống nếu train mới)
RESUME_FROM = ""  # Ví dụ: "/content/drive/MyDrive/Traffic-MARL-Training/checkpoints/gat_marl_mydinh_best.pt"

print(f"📋 Config: {MODEL} | {NUM_WORKERS} worker(s) | {EPISODES} episodes")
print(f"   Obstacle: prob={OBSTACLE_PROB}, max={OBSTACLE_MAX}")
print(f"   Resume: {'Yes → ' + RESUME_FROM if RESUME_FROM else 'Fresh train'}")

---
## 5. 🚀 Train

In [ ]:
import subprocess
import shutil
import time
import os

os.chdir(REPO_DIR)

# ── Build command ────────────────────────────────────────────────
cmd = [
    "python", "-m", "training.train_parallel",
    "--model", MODEL,
    "--num-workers", str(NUM_WORKERS),
    "--episodes", str(EPISODES),
    "--delta-time", str(DELTA_TIME),
    "--obstacle-prob", str(OBSTACLE_PROB),
    "--obstacle-max-count", str(OBSTACLE_MAX),
    "--obstacle-duration-min", str(OBSTACLE_DUR_MIN),
    "--obstacle-duration-max", str(OBSTACLE_DUR_MAX),
]

if RESUME_FROM:
    cmd += ["--resume", RESUME_FROM]

print(f"🚀 Command: {' '.join(cmd)}")
print("=" * 60)

# ── Run training ─────────────────────────────────────────────────
env = os.environ.copy()
env["SUMO_HOME"] = "/usr/share/sumo"
env["PYTHONUNBUFFERED"] = "1"  # real-time output

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
    cwd=REPO_DIR,
)

# Stream output real-time + periodic Drive sync
last_sync = time.time()
SYNC_INTERVAL = 300  # sync lên Drive mỗi 5 phút

try:
    for line in process.stdout:
        print(line, end="")

        # Auto-sync checkpoint + log lên Drive định kỳ
        if time.time() - last_sync > SYNC_INTERVAL:
            # Sync checkpoints
            ckpt_src = os.path.join(REPO_DIR, "checkpoints")
            if os.path.exists(ckpt_src):
                shutil.copytree(
                    ckpt_src,
                    f"{DRIVE_DIR}/checkpoints",
                    dirs_exist_ok=True,
                )
            # Sync logs
            log_src = os.path.join(REPO_DIR, "logs")
            if os.path.exists(log_src):
                shutil.copytree(
                    log_src,
                    f"{DRIVE_DIR}/logs",
                    dirs_exist_ok=True,
                )
            last_sync = time.time()
            print(f"  💾 [Auto-sync] Saved to Drive ({DRIVE_DIR})")

    process.wait()
    print("\n" + "=" * 60)
    print(f"✅ Training finished! Exit code: {process.returncode}")

except KeyboardInterrupt:
    print("\n⚠️ Training interrupted by user")
    process.terminate()
    process.wait(timeout=30)

---
## 6. 💾 Lưu kết quả lên Google Drive

In [ ]:
import shutil
import os

os.chdir(REPO_DIR)

# ── Sync checkpoints ──────────────────────────────────────────────
ckpt_src = os.path.join(REPO_DIR, "checkpoints")
if os.path.exists(ckpt_src):
    shutil.copytree(ckpt_src, f"{DRIVE_DIR}/checkpoints", dirs_exist_ok=True)
    # List saved files
    for root, dirs, files in os.walk(f"{DRIVE_DIR}/checkpoints"):
        for f in files:
            fpath = os.path.join(root, f)
            size_mb = os.path.getsize(fpath) / 1024 / 1024
            print(f"  📦 {os.path.relpath(fpath, DRIVE_DIR)} ({size_mb:.1f} MB)")
else:
    print("⚠️ No checkpoints found")

# ── Sync logs ──────────────────────────────────────────────────────
log_src = os.path.join(REPO_DIR, "logs")
if os.path.exists(log_src):
    shutil.copytree(log_src, f"{DRIVE_DIR}/logs", dirs_exist_ok=True)
    for root, dirs, files in os.walk(f"{DRIVE_DIR}/logs"):
        for f in files:
            fpath = os.path.join(root, f)
            size_kb = os.path.getsize(fpath) / 1024
            print(f"  📊 {os.path.relpath(fpath, DRIVE_DIR)} ({size_kb:.1f} KB)")
else:
    print("⚠️ No logs found")

print(f"\n✅ All results saved to: {DRIVE_DIR}")

---
## 7. 📊 Quick Preview — Training Log

In [ ]:
import csv
import os

# Tìm log file
log_candidates = [
    f"{DRIVE_DIR}/logs/mydinh/{MODEL}/training_log.csv",
    f"{REPO_DIR}/logs/mydinh/{MODEL}/training_log.csv",
]

log_file = None
for path in log_candidates:
    if os.path.exists(path):
        log_file = path
        break

if log_file:
    with open(log_file, 'r') as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    print(f"📊 Training log: {len(rows)} episodes")
    print(f"   File: {log_file}")
    print("\n" + "=" * 80)
    print(f"{'Ep':>5} | {'Reward':>10} | {'Speed':>8} | {'Wait':>8} | {'Loss':>10} | {'LR':>12}")
    print("-" * 80)

    # Show first 5 + last 10
    display_rows = rows[:5] + [{"episode": "..."}] + rows[-10:] if len(rows) > 15 else rows
    for r in display_rows:
        if r.get("episode") == "...":
            print(f"{'...':>5} |{'':>10} |{'':>8} |{'':>8} |{'':>10} |{'':>12}")
        else:
            print(
                f"{r.get('episode','?'):>5} | "
                f"{float(r.get('global_reward', 0)):>10.2f} | "
                f"{float(r.get('avg_speed', 0)):>7.1f}  | "
                f"{float(r.get('avg_waiting_time', 0)):>7.1f}  | "
                f"{float(r.get('loss', 0)):>10.6f} | "
                f"{float(r.get('learning_rate', 0)):>12.7f}"
            )

    print("=" * 80)
    # Summary
    rewards = [float(r['global_reward']) for r in rows if 'global_reward' in r]
    lrs = [float(r['learning_rate']) for r in rows if 'learning_rate' in r]
    if rewards:
        print(f"\n📈 Reward range: {min(rewards):.2f} → {max(rewards):.2f}")
        print(f"   Best reward:  {max(rewards):.2f}")
    if lrs:
        print(f"   LR range:     {min(lrs):.7f} → {max(lrs):.7f}")
else:
    print("⚠️ No training log found. Train first!")

---
## 8. ⬇️ Download trực tiếp (không qua Drive)

In [ ]:
import shutil
import os
from google.colab import files

os.chdir(REPO_DIR)

# Đóng gói checkpoints + logs thành 1 file zip
zip_name = f"{MODEL}_training_results"
zip_path = f"/content/{zip_name}"

# Tạo temp dir chứa cả checkpoints + logs
tmp_dir = f"/content/_pack_{MODEL}"
if os.path.exists(tmp_dir):
    shutil.rmtree(tmp_dir)
os.makedirs(tmp_dir)

# Copy checkpoints
ckpt_src = os.path.join(REPO_DIR, "checkpoints")
if os.path.exists(ckpt_src):
    shutil.copytree(ckpt_src, f"{tmp_dir}/checkpoints", dirs_exist_ok=True)

# Copy logs
log_src = os.path.join(REPO_DIR, "logs")
if os.path.exists(log_src):
    shutil.copytree(log_src, f"{tmp_dir}/logs", dirs_exist_ok=True)

# Zip
shutil.make_archive(zip_path, "zip", tmp_dir)
zip_file = f"{zip_path}.zip"
size_mb = os.path.getsize(zip_file) / 1024 / 1024
print(f"📦 {zip_name}.zip ({size_mb:.1f} MB)")

# Download
files.download(zip_file)
print("✅ Download started!")

# Cleanup
shutil.rmtree(tmp_dir, ignore_errors=True)